# Use the Pretrained BT Super-Resolution Models

This notebook is the beginner-friendly path for users who mainly want to load a released model, read an HDF5 brightness-temperature file, and generate a high-resolution prediction. It uses the `.keras` release artifacts through the repository loader so Kelvin normalization is handled for you.

## 1. Setup

Run this notebook from the repository root, or update `REPO_ROOT` below. The model files should be downloaded from the GitHub Release and placed in `release_assets/`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

from bt_super_resolution import load_keras_generator

## 2. Load a Released Keras Model

Use the Composite SSIM RRDN model for a conservative reconstruction-first result, or switch to `configs/rrdn_gan_batchnorm.yaml` for the GAN generator.

In [ ]:
CONFIG = REPO_ROOT / "configs/rrdn_composite_ssim_alpha_0.8.yaml"
# CONFIG = REPO_ROOT / "configs/rrdn_gan_batchnorm.yaml"

bundle = load_keras_generator(CONFIG, repository_root=REPO_ROOT)
print(bundle.name)

## 3. HDF5 Helpers

The repository supports files where the low-resolution brightness temperature is stored as `L/bt` or `bt`. Some files also include latitude and longitude arrays; these are optional for prediction but useful for plotting.

In [ ]:
LR_KEYS = ("L/bt", "bt")

def find_lr_key(handle):
    for key in LR_KEYS:
        if key in handle:
            return key
    raise KeyError(f"No LR BT dataset found. Checked: {LR_KEYS}")

def read_lr_bt(path):
    with h5py.File(path, "r") as handle:
        key = find_lr_key(handle)
        bt = np.asarray(handle[key][:], dtype=np.float32)
    return bt

def first_h5(path):
    path = Path(path)
    if path.is_file():
        return path
    files = sorted([*path.rglob("*.h5"), *path.rglob("*.hdf5")]) if path.exists() else []
    return files[0] if files else None

def plot_lr_and_prediction(lr_bt, prediction_bt, title):
    lr_2d = lr_bt[0] if lr_bt.ndim == 3 else lr_bt
    pred_2d = prediction_bt[0] if prediction_bt.ndim == 3 else prediction_bt
    vmin, vmax = np.nanpercentile(np.concatenate([lr_2d.ravel(), pred_2d.ravel()]), [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, panel, label in zip(axes, [lr_2d, pred_2d], ["Input LR", "Predicted HR"]):
        image = ax.imshow(panel, cmap="turbo", vmin=vmin, vmax=vmax)
        ax.set_title(label)
        ax.set_axis_off()
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Brightness Temp. (K)")
    fig.suptitle(title)
    fig.tight_layout()
    return fig

## 4. Run the Public AMSR2 Example

This file is intentionally small enough to keep in Git and is useful for checking that the model and environment work.

In [ ]:
amsr2_example = REPO_ROOT / "sample_data/amsr2_example.h5"
lr_bt = read_lr_bt(amsr2_example)
prediction_bt = bundle.predict_kelvin(lr_bt, batch_size=1)

print("LR shape:", lr_bt.shape)
print("Prediction shape:", prediction_bt.shape)
plot_lr_and_prediction(lr_bt, prediction_bt, amsr2_example.name);

## 5. Optional Local Sensor Collections

The following cells run only if the corresponding local folders exist. These larger folders should not be committed to Git. They are examples for users who have their own compatible HDF5 files.

In [ ]:
sensor_roots = {
    "AMSR2 local collection": REPO_ROOT / "AMSR2",
    "ATMS local collection": REPO_ROOT / "ATMS",
    "Tomorrow.io local collection": REPO_ROOT / "tomorrow.io.new",
}

for label, root in sensor_roots.items():
    example = first_h5(root)
    if example is None:
        print(f"Skipping {label}: {root} not found or contains no HDF5 files.")
        continue
    print(f"\n{label}: {example}")
    lr_bt = read_lr_bt(example)
    prediction_bt = bundle.predict_kelvin(lr_bt, batch_size=1)
    print("LR shape:", lr_bt.shape, "Prediction shape:", prediction_bt.shape)
    plot_lr_and_prediction(lr_bt, prediction_bt, example.name)
    plt.show()

## 6. Save a Prediction

For batch prediction and HDF5 output, the command-line script is usually better than writing notebook output by hand:

```bash
python scripts/inference/make_prediction.py \
    --input sample_data/amsr2_example.h5 \
    --output outputs/example_keras_predictions.h5 \
    --artifact-format keras \
    --batch-size 1 \
    --overwrite \
    --strict
```

Then plot the saved prediction with longitude/latitude axes when geolocation variables are available:

```bash
python scripts/evaluation/plot_predictions.py \
    --input outputs/example_keras_predictions.h5 \
    --output-dir outputs/example_plots \
    --overwrite \
    --strict
```